# Finanzberater_Agent: zwei Agenten in einer Schleife

Ein begeisterter **Berater** schreibt eine Antwort an einen Bankkunden.
Eine **Compliance-Prüfung** kontrolliert sie. Wenn etwas nicht stimmt, muss der Berater
nachbessern. Das wiederholt sich, bis alles in Ordnung ist (höchstens 3 Runden).

Geprüft wird auf zwei Arten:
- **Harte Regeln** prüft Python: Länge, Wort «Risiko», keine Prozentzahlen. Das ist immer gleich und kostet nichts.
- **Weiche Regeln** prüft ein Sprachmodell: kein Renditeversprechen, Hinweis auf ein Beratungsgespräch.

## Vorbereitung (einmal)
1. **Kopie speichern:** *Datei → Kopie in Drive speichern*. Sonst gehen deine Änderungen verloren.
2. **API-Key hinterlegen:** links auf das **Schlüssel-Symbol** (Secrets) klicken,
   *Neues Secret hinzufügen*, Name `API_KEY`, als Wert den Key von Moodle einfügen,
   **Notebook-Zugriff** einschalten. Der Key steht so nie im Notebook selbst.

## Jedes Mal
Colab löscht nach einer Pause alles. Führe die Zellen darum **von oben nach unten** aus
(Play-Knopf links an jeder Zelle, oder *Laufzeit → Alle ausführen*).

> **Tipp für den Code-Assistenten Gemini:** Schreib in deine Anfrage dazu:
> *«Nutze Google ADK, Modell `gemini-3.5-flash`, halte den Code so einfach wie möglich,
> Kommentare auf Deutsch.»*

## 1. Einrichten

In [ ]:
# Google ADK (Agent Development Kit) installieren. Dauert ca. 1 Minute.
!pip install -q "google-adk>=2.15,<3"

In [ ]:
# API-Key aus den Colab-Secrets holen (Schlüssel-Symbol links).
import os
from google.colab import userdata

# ADK sucht den Key unter dem Namen GOOGLE_API_KEY.
os.environ["GOOGLE_API_KEY"] = userdata.get("API_KEY")
print("Key geladen")

## 2. Harte Regeln in Python

Diese Funktion ist ein **Callback**: ADK ruft sie automatisch auf, hier jedes Mal,
**nachdem** der Berater seinen Entwurf geschrieben hat (`after_agent_callback`).

Der **State** ist ein gemeinsamer Notizblock aller Agenten. Die Funktion liest dort den
Entwurf und schreibt ihr Ergebnis unter `regelcheck` hinein.

In [ ]:
import re

from google.adk.agents.callback_context import CallbackContext


def regeln_pruefen(callback_context: CallbackContext) -> None:
    """after_agent_callback: läuft NACHDEM der Berater seinen Entwurf geschrieben hat."""
    entwurf = callback_context.state.get("entwurf", "")
    verstoesse = []

    if len(entwurf.split()) > 80:
        verstoesse.append(f"zu lang ({len(entwurf.split())} statt höchstens 80 Wörter)")
    if "risiko" not in entwurf.lower():
        verstoesse.append("das Wort 'Risiko' fehlt")
    if re.search(r"\d+\s?%", entwurf):  # eine Zahl mit Prozentzeichen
        verstoesse.append("enthält eine Prozentzahl (wirkt wie ein Renditeversprechen)")

    # Das Ergebnis kommt in den State. Der Prüfer liest es dort als {regelcheck}.
    callback_context.state["regelcheck"] = "; ".join(verstoesse) if verstoesse else "OK"
    return None  # None heisst: die Antwort des Beraters bleibt unverändert

## 3. Die beiden Agenten

- `output_key="entwurf"`: Die Antwort des Agenten wird im State unter `entwurf` gespeichert.
- `{entwurf}` in einer Instruktion: Hier setzt ADK den Wert aus dem State ein.
- `{kritik?}` mit Fragezeichen: Platzhalter, der fehlen darf. In Runde 1 gibt es noch keine Kritik.
- `exit_loop`: ein fertiges Tool von ADK. Ruft der Prüfer es auf, endet die Schleife.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import exit_loop

berater = Agent(
    name="Berater_Agent",
    model="gemini-3.5-flash",
    instruction=(
        "Du bist ein begeisterter Kundenberater einer Schweizer Bank. "
        "Beantworte die Kundenfrage auf Deutsch, motivierend und mit konkreten Zahlen.\n"
        "Kritik der Compliance zur letzten Version (leer in Runde 1): {kritik?}\n"
        "Wenn es Kritik gibt, überarbeite deinen Text so, dass alle Punkte erfüllt sind. "
        "Gib nur den Antworttext an den Kunden aus."
    ),
    output_key="entwurf",
    after_agent_callback=regeln_pruefen,
)

pruefer = Agent(
    name="Pruefer_Agent",
    model="gemini-3.5-flash",
    instruction=(
        "Du bist die Compliance-Abteilung einer Schweizer Bank und prüfst diesen Entwurf:\n"
        "{entwurf}\n\n"
        "Ergebnis der automatischen Regelprüfung: {regelcheck}\n\n"
        "Prüfe zusätzlich: kein Renditeversprechen, keine persönliche Anlageempfehlung "
        "ohne Beratungsgespräch, ein Hinweis auf ein Beratungsgespräch ist enthalten.\n"
        "Wenn die Regelprüfung OK ist und du keinen Verstoss findest: rufe exit_loop auf.\n"
        "Sonst: nenne die Verstösse in einer kurzen Liste, ohne den Text selbst umzuschreiben."
    ),
    tools=[exit_loop],
    output_key="kritik",
)

## 4. Die Schleife

Ein **LoopAgent** führt seine Unteragenten immer wieder der Reihe nach aus:
Berater, Prüfer, Berater, Prüfer, ... bis `exit_loop` aufgerufen wird
oder `max_iterations` erreicht ist.

Hier entscheidet also **der Code** über den Ablauf, nicht das Sprachmodell.

*(Die Warnung «LoopAgent is deprecated» kannst du ignorieren. Es funktioniert,
der Nachfolger heisst `Workflow`.)*

In [ ]:
from google.adk.agents import LoopAgent

root_agent = LoopAgent(
    name="Finanzberater_Agent",
    sub_agents=[berater, pruefer],
    max_iterations=3,  # Notbremse, falls sich die beiden nie einig werden
)

## 5. Eine Kundenfrage stellen

Jedes Mal, wenn du diese Zelle ausführst, beginnt ein neues Gespräch mit leerem State.
Achte darauf: Wie viele Runden braucht es? Was bemängelt der Prüfer?

In [ ]:
from google.adk.runners import InMemoryRunner

runner = InMemoryRunner(agent=root_agent)
await runner.run_debug(
    "Ich habe 20'000 Franken gespart. Wo soll ich die anlegen, damit sie sich schnell vermehren?",
    verbose=True,
)

## 6. In den State schauen

So sieht der gemeinsame Notizblock nach dem letzten Durchlauf aus.
`kritik` zeigt die letzte Kritik: Wenn der Prüfer am Schluss `exit_loop` aufruft, schreibt er keine neue.

In [ ]:
session = await runner.session_service.get_session(
    app_name=runner.app_name, user_id="debug_user_id", session_id="debug_session_id"
)
for name, wert in session.state.items():
    print(f"--- {name} ---\n{wert}\n")

## 7. Ausprobieren

Nach jeder Änderung die geänderte Zelle und alle Zellen darunter **nochmal ausführen**
(oder *Laufzeit → Alle ausführen*).

1. Setze `max_iterations=1`. Was passiert, wenn der Entwurf nicht durchkommt?
2. Verschärfe eine harte Regel, z.B. höchstens 40 statt 80 Wörter.
3. Füge eine eigene harte Regel hinzu, z.B. dass das Wort «garantiert» verboten ist.
4. Lass den Berater frecher werden («verspreche hohe Gewinne») und schau, ob die Prüfung hält.